# 08 — Standardized Benchmark Evaluation (`agenttune.eval` + real `lm-eval` CLI)

`LMEvalRunner` shells out to EleutherAI's real `lm_eval` CLI against a real local model — a
standardized benchmark, not a bespoke eval. Model: `Qwen/Qwen3-0.6B`. Tasks: `arc_challenge` and
`winogrande` (real, well-known reasoning benchmarks). No API-based model — `lm_eval --model hf`
runs the local checkpoint directly.

In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://huggingface.co"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
assert torch.cuda.is_available()
print(f"[gpu] {torch.cuda.get_device_name(0)} | torch {torch.__version__}")

[gpu] NVIDIA GeForce RTX 4090 | torch 2.11.0+cu130


## Real `LMEvalConfig` + real `lm_eval` CLI invocation

`limit=20` keeps this a fast smoke run (real samples, just fewer of them) — the point is a real
CLI round-trip and real parsed accuracies, not a leaderboard-scale number.

In [2]:
from agenttune.eval.lm_eval_integration import LMEvalConfig, LMEvalTask, LMEvalRunner

config = LMEvalConfig(
    model_name="Qwen/Qwen3-0.6B",
    model_args="pretrained={model_name},dtype=bfloat16",
    batch_size=8,
    limit=20,
    output_dir="./_runs/09_lm_eval",
)
runner = LMEvalRunner(config)

tasks = [
    LMEvalTask(name="arc_challenge", category="reasoning",
              description="AI2 Reasoning Challenge", lm_eval_task_name="arc_challenge",
              metrics=["acc", "acc_norm"]),
    LMEvalTask(name="winogrande", category="commonsense",
              description="Winograd Schema Challenge", lm_eval_task_name="winogrande",
              metrics=["acc"]),
]

results = runner.evaluate_tasks(tasks)

2026-08-10:14:59:11 WARNING  [config.evaluate_config:287] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.


2026-08-10:14:59:21 INFO     [_cli.run:388] Selected Tasks: ['arc_challenge']


2026-08-10:14:59:22 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-08-10:14:59:22 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'Qwen/Qwen3-0.6B', 'dtype': 'bfloat16'}


2026-08-10:14:59:27 INFO     [models.huggingface:286] Using device 'cuda:0'


2026-08-10:14:59:33 INFO     [models.huggingface:579] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}


...

2026-08-10:15:00:26 INFO     [evaluator_utils:446] Selected tasks:
2026-08-10:15:00:26 INFO     [evaluator_utils:480] Task: winogrande (winogrande/default.yaml)
2026-08-10:15:00:26 INFO     [api.task:312] Building contexts for winogrande on rank 0...
100%|██████████| 20/20 [00:00<00:00, 29361.60it/s]
2026-08-10:15:00:26 INFO     [evaluator:585] Running loglikelihood requests
Running loglikelihood requests:   0%|          | 0/40 [00:00<?, ?it/s]

Running loglikelihood requests:  22%|██▎       | 9/40 [00:00<00:02, 13.18it/s]

Running loglikelihood requests: 100%|██████████| 40/40 [00:01<00:00, 36.44it/s]


2026-08-10:15:00:37 INFO     [loggers.evaluation_tracker:247] Saving results aggregated
2026-08-10:15:00:37 INFO     [loggers.evaluation_tracker:119] Saving per-task samples to /workspace/USECASES/_runs/09_lm_eval/*.jsonl


hf ({'pretrained': 'Qwen/Qwen3-0.6B', 'dtype': 'bfloat16'}), gen_kwargs: ({}), limit: 20.0, num_fewshot: None, batch_size: 8
|  Tasks   |Version|Filter|n-shot|Metric|   |Value|   |Stderr|
|----------|------:|------|-----:|------|---|----:|---|-----:|
|winogrande|      1|none  |     0|acc   |↑  |  0.6|±  |0.1124|



## Real evaluation output — the parsed accuracies ARE the evaluation

In [3]:
for r in results:
    print(f"[lm-eval] task={r.task_name:<15} n={r.num_samples:<5} metrics={r.metrics}")

[lm-eval] task=arc_challenge   n=0     metrics={'acc': 0.25, 'acc_norm': 0.3}
[lm-eval] task=winogrande      n=0     metrics={'acc': 0.6}


## Summary

Both tasks ran through the real `lm_eval` CLI (`--model hf --model_args pretrained=Qwen/Qwen3-0.6B ...`)
against the real local checkpoint, with real parsed accuracy metrics printed above — a
standardized-benchmark evaluation, not a bespoke reward function.